# 13. 신도시 반사실 (과제 4 가이드라인) — 킬링포인트 F

판교·분당(개발완료)에서 **용도(blockType)→열스트레스** 관계를 학습 →
창릉·왕숙 **계획 blockType(05)**에 적용 → **개발 후 예상 HSI** 예측.

- 현재(미개발, 시원) vs 개발후(예측) 비교 → 개발로 인한 열 상승 정량화
- 예측 고스트레스 계획블록 도출 → 설계 수정 제안(녹지·수공간·그늘)
- ⚠️ 가정: 판교·분당 용도→열 관계가 북부 신도시에 전이(상대열은 전이, 절대값은 주의)

산출물: `data/results/newtown_counterfactual.geojson`, `output/figures/fig15_newtown.png`

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, geopandas as gpd
from src.config import DATA_RESULTS, OUT_FIGURES, CRS_SUBMIT

g = gpd.read_file(DATA_RESULTS / "heat_index_final.geojson")
DEV = ["성남판교지구 택지개발사업", "성남분당"]
NEW = ["고양창릉 공공주택지구", "남양주왕숙 공공주택지구"]

# 학습: 개발지 용도별 HSI (표본 30격자 이상만)
dev = g[g.zoneName.isin(DEV)]
lk = dev.groupby("blockType")["hsi"].agg(["mean", "count"])
lk = lk[lk["count"] >= 30]["mean"]
print("학습된 용도별 HSI (개발지) 상위/하위:")
print("  뜨거운 용도:", lk.nlargest(5).round(2).to_dict())
print("  시원한 용도:", lk.nsmallest(4).round(2).to_dict())

학습된 용도별 HSI (개발지) 상위/하위:
  뜨거운 용도: {'근린생활시설용지': 0.9, '중심상업': 0.89, '일반광장': 0.88, '사회복지시설': 0.87, '집단에너지공급시설': 0.87}
  시원한 용도: {'가압장': 0.19, '경관녹지': 0.24, '근린공원': 0.25, '녹지': 0.33}


In [2]:
# 적용: 신도시 계획 blockType → 예상 HSI
new = g[g.zoneName.isin(NEW)].copy()
new["hsi_future"] = new["blockType"].map(lk).fillna(dev["hsi"].mean())
print("신도시 현재(미개발) vs 개발후(예측) 평균 HSI:")
for z in NEW:
    sub = new[new.zoneName == z]
    print(f"  {z}: 현재 {sub.hsi.mean():.2f} → 개발후 {sub.hsi_future.mean():.2f}  (+{sub.hsi_future.mean()-sub.hsi.mean():.2f})")
print("\n개발후 고스트레스 계획용도 (면적 큰 것 중):")
vc = new.groupby("blockType").agg(n=("hsi_future", "size"), future=("hsi_future", "mean"))
print(vc[vc.n >= 50].nlargest(6, "future")["future"].round(2).to_dict())

신도시 현재(미개발) vs 개발후(예측) 평균 HSI:
  고양창릉 공공주택지구: 현재 0.22 → 개발후 0.65  (+0.43)
  남양주왕숙 공공주택지구: 현재 0.44 → 개발후 0.67  (+0.23)

개발후 고스트레스 계획용도 (면적 큰 것 중):
{'근린생활시설용지': 0.9, '집단에너지공급시설': 0.87, '주차장': 0.85, '단독주택': 0.84, '교통광장': 0.81, '도로': 0.8}


In [3]:
new["hsi_rise"] = new["hsi_future"] - new["hsi"]
new[["grid_id", "zoneName", "blockType", "hsi", "hsi_future", "hsi_rise", "geometry"]].to_crs(CRS_SUBMIT).to_file(
    DATA_RESULTS / "newtown_counterfactual.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

import matplotlib.pyplot as plt
from src.viz.fonts import setup_font
setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
fig, ax = plt.subplots(1, 3, figsize=(20, 7))
for a, col, ttl in [(ax[0], "hsi", "현재 HSI (미개발)"), (ax[1], "hsi_future", "개발후 예측 HSI"),
                    (ax[2], "hsi_rise", "열 상승폭 (설계 개입 우선)")]:
    cmap = "YlOrRd" if col != "hsi_rise" else "Reds"
    new.plot(column=col, cmap=cmap, ax=a, linewidth=0, legend=True, legend_kwds={"shrink": 0.5},
             vmin=0, vmax=1 if col != "hsi_rise" else new.hsi_rise.quantile(0.95))
    a.set_title(ttl); a.set_axis_off()
plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig15_newtown.png", dpi=120); plt.close()
print("저장 완료. 열상승 큰 계획블록 = 설계단계 녹지·수공간·그늘 우선 투입 대상.")

저장 완료. 열상승 큰 계획블록 = 설계단계 녹지·수공간·그늘 우선 투입 대상.
